# 개별종목 조합E — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.4250,0.5012,-0.0762,0.3360,0.2225,0.3054
1,2,NaN,980,20150123,20150421,0.3788,0.3978,-0.0190,0.3437,0.2393,0.3084
2,3,balanced,1210,20151228,20160328,0.3422,0.3762,-0.0339,0.3397,0.3223,0.3345
3,4,balanced,1439,20161202,20170228,0.3743,0.4617,-0.0874,0.3329,0.2757,0.3225
4,5,balanced,1669,20171113,20180207,0.3577,0.3901,-0.0324,0.3459,0.3072,0.3355
5,6,balanced,1899,20181024,20190118,0.3757,0.3725,0.0032,0.3746,0.3593,0.3697
6,7,balanced,2129,20190930,20191224,0.4051,0.4781,-0.0731,0.3584,0.3035,0.3507
7,8,balanced,2359,20200902,20201130,0.3588,0.3476,0.0112,0.3581,0.3812,0.3657
8,9,balanced,2589,20210806,20211105,0.3652,0.3916,-0.0264,0.3569,0.3307,0.3503
9,10,balanced,2818,20220714,20221012,0.3571,0.3454,0.0117,0.3567,0.3217,0.3444


,OOS 폴드 평균
accuracy,0.3741
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0228
macro_f1,0.3537
down_recall,0.3148
core_harmonic_mean,0.3435


재실행 명령: python scripts/run_stock_model_experiment.py
